# GeoSentinel-AI Cloud Training Notebook
This notebook is designed for training the 150-epoch Change Detection model on free cloud GPUs (Google Colab / Kaggle).

## 1. Setup Environment

In [ ]:
!pip install torch torchvision torchgeo lightning segmentation-models-pytorch rasterio pystac-client planetary-computer

## 2. Clone Repository (If running on Colab)
*Skip this cell if you are running as a Kaggle script where the repo is already included.*

In [ ]:
import os
if not os.path.exists('GeoSentinel-AI'):
    !git clone https://github.com/karthikeya-bhamidipati/GeoSentinel-AI.git
os.chdir('GeoSentinel-AI')

## 3. Mount Google Drive (Colab Only)
This ensures your weights are permanently saved to your Google Drive during training, preventing data loss if Colab disconnects.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    os.makedirs('data/weights', exist_ok=True)
    # Copy existing deeplab weights if available
    if os.path.exists('/content/drive/MyDrive/deeplabv3plus_best.pt'):
        shutil.copy('/content/drive/MyDrive/deeplabv3plus_best.pt', 'data/weights/')
except ImportError:
    print("Not running on Google Colab.")

## 4. Train the Model

**For Semantic Model (Our USP):** Set `ablation_mode = False`

**For Baseline ResNet (Control):** Set `ablation_mode = True`

In [ ]:
epochs = 150
batch_size = 4  # Adjust based on GPU VRAM (Colab T4 can handle 4-8)
ablation_mode = False # SET TO TRUE FOR THE BASELINE STUDY

cmd = f"python scripts/train_change.py --epochs {epochs} --batch-size {batch_size}"
if ablation_mode:
    cmd += " --ablation"

!{cmd}

## 5. Save Final Weights (Colab Only)

In [ ]:
try:
    import shutil
    if ablation_mode:
        shutil.copy('data/weights/change_unet_baseline_best.pt', '/content/drive/MyDrive/')
    else:
        shutil.copy('data/weights/change_unet_best.pt', '/content/drive/MyDrive/')
    print("Weights successfully backed up to Google Drive!")
except ImportError:
    pass